# 09_train_tactical_vision.ipynb — TacticalVisionNet Training Pipeline

This notebook implements the training loop, loss function, and metrics evaluation for the custom `TacticalVisionNet` model, fulfilling the specifications defined in [SPEC.md](file:///Users/jpswaynos/Git/urban-palm-tree/SPEC.md).

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src'))

import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

from inference.tactical_vision_net import TacticalVisionNet

print("Imports completed successfully!")

In [ ]:
class TacticalDataset(Dataset):
    def __init__(self, json_path, transform=None):
        self.json_path = Path(json_path)
        with open(self.json_path, 'r') as f:
            self.records = json.load(f)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        main_img = Image.open(rec['main_image_path']).convert('RGB')
        minimap_img = Image.open(rec['minimap_image_path']).convert('RGB')

        if self.transform:
            main_img = self.transform(main_img)
            minimap_img = self.transform(minimap_img)

        possession = torch.tensor(rec['possession'], dtype=torch.long)
        zone = torch.tensor(rec['zone'], dtype=torch.long)
        ball = torch.tensor(rec['ball'], dtype=torch.float)

        return main_img, minimap_img, possession, zone, ball

In [ ]:
class MultiTaskLoss(nn.Module):
    def __init__(self, w_possession=1.0, w_zone=1.0, w_ball=10.0):
        super().__init__()
        self.w_pos = w_possession
        self.w_zone = w_zone
        self.w_ball = w_ball
        self.criterion_pos = nn.CrossEntropyLoss()
        self.criterion_zone = nn.CrossEntropyLoss()
        self.criterion_ball = nn.MSELoss()

    def forward(self, pred, target_pos, target_zone, target_ball):
        loss_pos = self.criterion_pos(pred['possession'], target_pos)
        loss_zone = self.criterion_zone(pred['zone'], target_zone)
        loss_ball = self.criterion_ball(pred['ball'], target_ball)
        
        total_loss = self.w_pos * loss_pos + self.w_zone * loss_zone + self.w_ball * loss_ball
        return total_loss, loss_pos, loss_zone, loss_ball

In [ ]:
def calculate_f1_score(preds, targets, num_classes=3):
    f1_list = []
    for c in range(num_classes):
        tp = sum((p == c and t == c) for p, t in zip(preds, targets))
        fp = sum((p == c and t != c) for p, t in zip(preds, targets))
        fn = sum((p != c and t == c) for p, t in zip(preds, targets))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        f1_list.append(f1)
    return sum(f1_list) / len(f1_list)

def calculate_l2_error(preds, targets):
    diff = preds - targets
    dist = torch.sqrt(torch.sum(diff ** 2, dim=1))
    return torch.mean(dist).item()

In [ ]:
# Setup transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
dataset_path = "../artifacts/collate_cache/matched_records.json"
dataset = TacticalDataset(dataset_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

print(f"Loaded dataset with {len(dataset)} records.")

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")

# Initialize model
model = TacticalVisionNet(pretrained=False).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
criterion = MultiTaskLoss()

# Single epoch simulation
model.train()
epoch_loss = 0.0
for batch_idx, (main_img, minimap_img, pos_label, zone_label, ball_label) in enumerate(dataloader):
    main_img = main_img.to(device)
    minimap_img = minimap_img.to(device)
    pos_label = pos_label.to(device)
    zone_label = zone_label.to(device)
    ball_label = ball_label.to(device)
    
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model(main_img, minimap_img)
    
    # Loss computation
    loss, l_pos, l_zone, l_ball = criterion(outputs, pos_label, zone_label, ball_label)
    
    # Backward pass and optimization
    loss.backward()
    optimizer.step()
    
    epoch_loss += loss.item()
    print(f"Batch {batch_idx+1}/{len(dataloader)} - Loss: {loss.item():.4f} (Pos: {l_pos.item():.4f}, Zone: {l_zone.item():.4f}, Ball: {l_ball.item():.4f})")

print(f"Epoch complete! Average loss: {epoch_loss / len(dataloader):.4f}")

In [ ]:
# Validation run simulation
model.eval()
all_pos_preds = []
all_pos_targets = []
all_zone_preds = []
all_zone_targets = []
all_ball_preds = []
all_ball_targets = []

with torch.no_grad():
    for main_img, minimap_img, pos_label, zone_label, ball_label in dataloader:
        main_img = main_img.to(device)
        minimap_img = minimap_img.to(device)
        
        outputs = model(main_img, minimap_img)
        
        # Predictions
        pos_preds = torch.argmax(outputs['possession'], dim=1).cpu().tolist()
        zone_preds = torch.argmax(outputs['zone'], dim=1).cpu().tolist()
        
        all_pos_preds.extend(pos_preds)
        all_pos_targets.extend(pos_label.tolist())
        
        all_zone_preds.extend(zone_preds)
        all_zone_targets.extend(zone_label.tolist())
        
        all_ball_preds.append(outputs['ball'].cpu())
        all_ball_targets.append(ball_label)

# Concatenate ball regression targets
ball_preds_tensor = torch.cat(all_ball_preds, dim=0)
ball_targets_tensor = torch.cat(all_ball_targets, dim=0)

# Calculate metrics
f1_pos = calculate_f1_score(all_pos_preds, all_pos_targets)
f1_zone = calculate_f1_score(all_zone_preds, all_zone_targets)
l2_ball = calculate_l2_error(ball_preds_tensor, ball_targets_tensor)

print("\n--- Validation Metrics ---")
print(f"Possession F1 Score (Macro): {f1_pos:.4f}")
print(f"Zone F1 Score (Macro):       {f1_zone:.4f}")
print(f"Ball Coordinates L2 Error:   {l2_ball:.4f}")